# 🐼 Pandas vs 🐻‍❄️ Polars — Speed Test on 2 Million Rows

**By a Pandas user trying Polars for the first time!**

I have been using Pandas for all my data projects.  
Recently I read an article saying Polars is much faster.  
So I decided to test it myself — no prior Polars experience, just curiosity!

---

### What I am testing:
- Same dataset, same 4 tasks
- Pandas way vs Polars way
- Which one is faster? (spoiler: the answer surprised me!)


## Step 1 — Install the libraries

First, let's make sure both libraries are installed.  
Pandas you probably already have. Polars needs to be installed.

In [ ]:
# Run this if you don't have polars installed
# !pip install polars pyarrow

# Check versions
import pandas as pd
import polars as pl

print("Pandas version :", pd.__version__)
print("Polars version :", pl.__version__)
print("Both libraries loaded successfully!")


## Step 2 — Create a Fake E-Commerce Dataset

I am creating a fake dataset that looks like real e-commerce orders.  
It has **2 million rows** — big enough to see a real speed difference.

| Column | What it means |
|---|---|
| order_id | unique ID for each order |
| user_id | who placed the order |
| product_id | which product was ordered |
| category | product category (Electronics, Books, etc.) |
| quantity | how many items ordered |
| price | price per item in ₹ |
| order_date | when the order was placed |


In [ ]:
import numpy as np

# Set a random seed so we get same data every time
np.random.seed(42)

# How many rows do we want?
n = 2_000_000  # 2 million rows

# Create some sample values to pick from
users    = [f"user_{i}"    for i in range(10000)]
products = [f"product_{i}" for i in range(500)]
categories = ["Electronics", "Clothing", "Food", "Books", "Sports", "Beauty", "Home"]

# Build the dataset as a dictionary
data = {
    "order_id"   : np.arange(n),
    "user_id"    : np.random.choice(users, n),
    "product_id" : np.random.choice(products, n),
    "category"   : np.random.choice(categories, n),
    "quantity"   : np.random.randint(1, 10, n),
    "price"      : np.round(np.random.uniform(10, 5000, n), 2),
    "order_date" : pd.date_range("2022-01-01", periods=n, freq="30s"),
}

# Create Pandas DataFrame
df_pandas = pd.DataFrame(data)

# Create Polars DataFrame from the same data
df_polars = pl.from_pandas(df_pandas)

print(f"Dataset created!")
print(f"Total rows  : {n:,}")
print(f"Total columns: {len(df_pandas.columns)}")
print()
print("Here is what the first 3 rows look like:")
df_pandas.head(3)


## Step 3 — Memory Usage Comparison

Before we even run any tasks, let's see how much RAM each library uses for the same data.


In [ ]:
pandas_memory = df_pandas.memory_usage(deep=True).sum() / 1024**2
polars_memory = df_polars.estimated_size() / 1024**2

print(f"Pandas memory usage : {pandas_memory:.1f} MB")
print(f"Polars memory usage : {polars_memory:.1f} MB")
print()
print(f"Polars uses {pandas_memory - polars_memory:.1f} MB LESS memory for the same data!")
print(f"That is a {(1 - polars_memory/pandas_memory)*100:.0f}% saving in RAM.")


## Task 1 — Total Revenue per Category

**The question:** How much total revenue did each category make?

This is a simple `groupby` + `sum` — something we do all the time in data analysis.


In [ ]:
import time

# ── Pandas way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_pandas_t1 = (
    df_pandas
    .groupby("category")["price"]   # group by category, pick price column
    .sum()                           # sum up all prices
    .reset_index()                   # make it a clean table again
    .rename(columns={"price": "total_revenue"})
)

pandas_time_t1 = time.perf_counter() - start

# ── Polars way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_polars_t1 = (
    df_polars
    .lazy()                          # polars: build the plan first, don't run yet
    .group_by("category")            # group by category
    .agg(
        pl.col("price").sum().alias("total_revenue")   # sum up prices
    )
    .collect()                       # NOW run everything
)

polars_time_t1 = time.perf_counter() - start

# ── Results ─────────────────────────────────────────────────────────────────
print(f"Task 1: Revenue per Category")
print(f"  Pandas  took : {pandas_time_t1:.3f} seconds")
print(f"  Polars  took : {polars_time_t1:.3f} seconds")
print(f"  Winner  : {'Polars' if polars_time_t1 < pandas_time_t1 else 'Pandas'} "
      f"({max(pandas_time_t1, polars_time_t1)/min(pandas_time_t1, polars_time_t1):.1f}x faster)")
print()
print("Pandas result:")
print(result_pandas_t1.sort_values("total_revenue", ascending=False))


## Task 2 — Top 10 Users by Total Spend

**The question:** Which 10 users spent the most money overall?

Here we need to calculate `price × quantity` first, then group, then sort.  
This involves more steps — let's see who handles it better.


In [ ]:
# ── Pandas way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_pandas_t2 = (
    df_pandas
    .assign(revenue = df_pandas["price"] * df_pandas["quantity"])  # new column = price x qty
    .groupby("user_id")["revenue"]    # group by user
    .sum()                             # total spend per user
    .nlargest(10)                      # keep only top 10
    .reset_index()
)

pandas_time_t2 = time.perf_counter() - start

# ── Polars way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_polars_t2 = (
    df_polars
    .lazy()
    .with_columns(
        (pl.col("price") * pl.col("quantity")).alias("revenue")   # new column = price x qty
    )
    .group_by("user_id")
    .agg(pl.col("revenue").sum())     # total spend per user
    .sort("revenue", descending=True) # sort highest first
    .limit(10)                        # keep only top 10
    .collect()
)

polars_time_t2 = time.perf_counter() - start

# ── Results ─────────────────────────────────────────────────────────────────
print(f"Task 2: Top 10 Users by Spend")
print(f"  Pandas  took : {pandas_time_t2:.3f} seconds")
print(f"  Polars  took : {polars_time_t2:.3f} seconds")
print(f"  Winner  : {'Polars' if polars_time_t2 < pandas_time_t2 else 'Pandas'} "
      f"({max(pandas_time_t2, polars_time_t2)/min(pandas_time_t2, polars_time_t2):.1f}x faster)")
print()
print("Pandas result (top 10 spenders):")
print(result_pandas_t2)


## Task 3 — Filter + Group (Electronics orders above ₹1000)

**The question:** Among Electronics products priced above ₹1000, which products were ordered the most?

This tests filtering first, then grouping — a very common real-world pattern.


In [ ]:
# ── Pandas way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

# Step 1: filter rows we want
electronics_df = df_pandas[
    (df_pandas["category"] == "Electronics") &   # only Electronics
    (df_pandas["price"] > 1000)                   # only price above 1000
]

# Step 2: group by product and sum quantities
result_pandas_t3 = (
    electronics_df
    .groupby("product_id")["quantity"]
    .sum()
    .reset_index()
    .sort_values("quantity", ascending=False)
    .head(10)
)

pandas_time_t3 = time.perf_counter() - start

# ── Polars way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_polars_t3 = (
    df_polars
    .lazy()
    .filter(
        (pl.col("category") == "Electronics") &   # only Electronics
        (pl.col("price") > 1000)                   # only price above 1000
    )
    .group_by("product_id")
    .agg(pl.col("quantity").sum())
    .sort("quantity", descending=True)
    .limit(10)
    .collect()
)

polars_time_t3 = time.perf_counter() - start

# ── Results ─────────────────────────────────────────────────────────────────
print(f"Task 3: Filter + Group (Electronics > Rs.1000)")
print(f"  Pandas  took : {pandas_time_t3:.3f} seconds")
print(f"  Polars  took : {polars_time_t3:.3f} seconds")
print(f"  Winner  : {'Polars' if polars_time_t3 < pandas_time_t3 else 'Pandas'} "
      f"({max(pandas_time_t3, polars_time_t3)/min(pandas_time_t3, polars_time_t3):.1f}x faster)")
print()
print("Top 10 Electronics products by quantity ordered:")
print(result_pandas_t3)


## Task 4 — Monthly Revenue Trend

**The question:** What was the total revenue for each month of 2022?

This requires extracting the month from a date column, then grouping.  
Date operations can be slow — let's see the difference here.


In [ ]:
# ── Pandas way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

# Make a copy so we don't change the original
df_pandas_copy = df_pandas.copy()

# Extract month from the date
df_pandas_copy["month"] = df_pandas_copy["order_date"].dt.to_period("M")

# Group by month and sum revenue
result_pandas_t4 = (
    df_pandas_copy
    .groupby("month")["price"]
    .sum()
    .reset_index()
    .rename(columns={"price": "total_revenue"})
)

pandas_time_t4 = time.perf_counter() - start

# ── Polars way ──────────────────────────────────────────────────────────────
start = time.perf_counter()

result_polars_t4 = (
    df_polars
    .lazy()
    .with_columns(
        pl.col("order_date").dt.truncate("1mo").alias("month")   # round date down to month
    )
    .group_by("month")
    .agg(pl.col("price").sum().alias("total_revenue"))
    .sort("month")
    .collect()
)

polars_time_t4 = time.perf_counter() - start

# ── Results ─────────────────────────────────────────────────────────────────
print(f"Task 4: Monthly Revenue Trend")
print(f"  Pandas  took : {pandas_time_t4:.3f} seconds")
print(f"  Polars  took : {polars_time_t4:.3f} seconds")
print(f"  Winner  : {'Polars' if polars_time_t4 < pandas_time_t4 else 'Pandas'} "
      f"({max(pandas_time_t4, polars_time_t4)/min(pandas_time_t4, polars_time_t4):.1f}x faster)")
print()
print("Monthly revenue (first 6 months):")
print(result_pandas_t4.head(6))


## Final Summary — Who Won Overall?

Let's put all results together in one clean table.


In [ ]:
# Collect all times
tasks   = ["Revenue per Category", "Top 10 Users by Spend",
           "Filter + Group", "Monthly Trend"]
pandas_times = [pandas_time_t1, pandas_time_t2, pandas_time_t3, pandas_time_t4]
polars_times = [polars_time_t1, polars_time_t2, polars_time_t3, polars_time_t4]

# Print summary table
print("=" * 62)
print(f"{'Task':<28} {'Pandas':>8} {'Polars':>8} {'Winner':>12}")
print("=" * 62)

for task, p, q in zip(tasks, pandas_times, polars_times):
    if p < q:
        winner = f"Pandas  ({p/q:.1f}x)"
    else:
        winner = f"Polars  ({q/p:.1f}x)"
    print(f"{task:<28} {p:>7.3f}s {q:>7.3f}s  {winner}")

print("=" * 62)
print()
print("Memory usage:")
print(f"  Pandas : {pandas_memory:.1f} MB")
print(f"  Polars : {polars_memory:.1f} MB")
print(f"  Polars saves {pandas_memory - polars_memory:.1f} MB ({(1 - polars_memory/pandas_memory)*100:.0f}% less RAM)")


## Bonus — Let's Visualize the Results!

A bar chart makes it easy to see where each library wins.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tasks_short = ["Revenue\nper Category", "Top 10\nUsers", "Filter +\nGroup", "Monthly\nTrend"]

x = np.arange(len(tasks_short))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars_pandas = ax.bar(x - bar_width/2, pandas_times, bar_width,
                     label="Pandas", color="#3266ad", alpha=0.85)
bars_polars = ax.bar(x + bar_width/2, polars_times, bar_width,
                     label="Polars", color="#1D9E75", alpha=0.85)

# Add value labels on top of each bar
for bar in bars_pandas:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}s", ha="center", va="bottom", fontsize=9)

for bar in bars_polars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}s", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Task", fontsize=12)
ax.set_ylabel("Time in seconds (lower is better)", fontsize=12)
ax.set_title("Pandas vs Polars — Speed Test on 2 Million Rows", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(tasks_short)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("pandas_vs_polars_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as pandas_vs_polars_benchmark.png")


## My Key Takeaways (as a Pandas beginner in Polars)

### What surprised me:
- **Pandas actually won 2 out of 4 tasks!** Nobody told me that.
- Polars was dramatically faster for the monthly trend task (date operations).
- Polars uses noticeably less RAM for the same data.

### Key difference in how they work:
| | Pandas | Polars |
|---|---|---|
| Execution | Runs each step immediately | Builds a plan first, then runs |
| CPU usage | Single core | All cores at once |
| Built in | Python/C | Rust (very fast language) |
| Syntax | `df.groupby()` | `df.lazy().group_by().collect()` |

### When should YOU use Polars?
- ✅ Dataset has millions of rows
- ✅ You do lots of date/time operations
- ✅ You are running out of RAM
- ❌ Small datasets — Pandas is fine and simpler

### My honest opinion:
> I am not switching completely to Polars yet.  
> But for big data tasks — especially date aggregations — I will definitely reach for it.  
> The `.lazy()` → `.collect()` pattern feels weird at first, but makes sense once you understand it means "plan first, execute later".

---
*Experiment done by a curious Pandas user | Dataset: 2 million fake e-commerce orders*
